In [1]:
import json
from dataclasses import dataclass, field
from typing import List, Dict

# ==========================================
# 1. Custom Exceptions
# ==========================================

class StudentRecordError(Exception):
    """Base exception for student record errors."""
    pass

class StudentNotFoundError(StudentRecordError):
    def __init__(self, student_id: str):
        self.student_id = student_id
        super().__init__(f"Student ID '{student_id}' not found in registry.")

class DuplicateStudentError(StudentRecordError):
    def __init__(self, student_id: str):
        self.student_id = student_id
        super().__init__(f"Student ID '{student_id}' already exists.")

class InvalidGradeError(StudentRecordError):
    def __init__(self, grade: float):
        self.grade = grade
        super().__init__(f"Grade '{grade}' is invalid. Must be between 0.0 and 100.0.")


# ==========================================
# 2. Data Models
# ==========================================

@dataclass
class CourseGrade:
    course_code: str
    course_name: str
    credits: float
    numeric_score: float

    def __post_init__(self):
        if not (0.0 <= self.numeric_score <= 100.0):
            raise InvalidGradeError(self.numeric_score)

    @property
    def gpa_points(self) -> float:
        if self.numeric_score >= 90: return 4.0
        elif self.numeric_score >= 80: return 3.0
        elif self.numeric_score >= 70: return 2.0
        elif self.numeric_score >= 60: return 1.0
        return 0.0

@dataclass
class Student:
    student_id: str
    name: str
    department: str
    grades: List[CourseGrade] = field(default_factory=list)

    def add_grade(self, course_code: str, course_name: str, credits: float, score: float):
        grade = CourseGrade(course_code, course_name, credits, score)
        self.grades.append(grade)

    @property
    def total_credits(self) -> float:
        return sum(g.credits for g in self.grades)

    @property
    def gpa(self) -> float:
        if not self.grades or self.total_credits == 0:
            return 0.0
        weighted_points = sum(g.gpa_points * g.credits for g in self.grades)
        return round(weighted_points / self.total_credits, 2)


# ==========================================
# 3. Manager Engine
# ==========================================

class StudentRecordManager:
    def __init__(self):
        self._students: Dict[str, Student] = {}

    def add_student(self, student_id: str, name: str, department: str) -> Student:
        if student_id in self._students:
            raise DuplicateStudentError(student_id)
        student = Student(student_id=student_id, name=name, department=department)
        self._students[student_id] = student
        return student

    def get_student(self, student_id: str) -> Student:
        if student_id not in self._students:
            raise StudentNotFoundError(student_id)
        return self._students[student_id]


# ==========================================
# 4. Execution & Output Verification
# ==========================================

if __name__ == '__main__':
    print("=== Student Record Library Test ===\n")
    manager = StudentRecordManager()

    # Add student
    s1 = manager.add_student("CS101", "Alice Smith", "Computer Science")
    s1.add_grade("CS50", "Intro to Programming", credits=4.0, score=92.5)
    s1.add_grade("MATH101", "Calculus I", credits=3.0, score=85.0)

    print(f"Student Registered : {s1.name} ({s1.student_id})")
    print(f"Department         : {s1.department}")
    print(f"Total Credits      : {s1.total_credits}")
    print(f"Calculated GPA     : {s1.gpa:.2f}\n")

    # Verify exception catching
    print("--- Testing Exception Traps ---")
    try:
        s1.add_grade("PHYS101", "Physics", credits=3.0, score=105.0)
    except InvalidGradeError as e:
        print(f"[CAUGHT] {e}")

    try:
        manager.get_student("CS999")
    except StudentNotFoundError as e:
        print(f"[CAUGHT] {e}") 

=== Student Record Library Test ===

Student Registered : Alice Smith (CS101)
Department         : Computer Science
Total Credits      : 7.0
Calculated GPA     : 3.57

--- Testing Exception Traps ---
[CAUGHT] Grade '105.0' is invalid. Must be between 0.0 and 100.0.
[CAUGHT] Student ID 'CS999' not found in registry.
